In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=0a09d32a6bd12ed0965783e17e77f95b4f8adcbf6f8a217db209db0e8902a9d4
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

1. Add Protocol Settings

In [2]:
backend = BasicSimulator()

# Protocol settings
N = 24                 # number of qubits Alice sends
TEST_FRACTION = 0.5     # proportion of sifted key publicly checked
ATTACK_THRESHOLD = 0.15 # if error rate is above this, report attack

2. Create Quantum Bit Generator

In [3]:
def quantum_random_bits(number_of_bits):
    # Generate random bits by measuring |+> = 1/sqrt(2)(|0> + |1>).
    circuit = QuantumCircuit(number_of_bits, number_of_bits)
    circuit.h(range(number_of_bits))
    circuit.measure(range(number_of_bits), range(number_of_bits))

    compiled = transpile(circuit, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    bitstring = list(result.get_counts().keys())[0]

    # Qiskit displays classical bits right-to-left, so reverse it for q0, q1, q2, ... order.
    return [int(bit) for bit in bitstring[::-1]]

3. Alice, Bob, Eve (Attacker) interaction for one Qubit

In [4]:
def alice_prepare_eve_measure_bob_measure(alice_bit, alice_basis, eve_basis, bob_basis):

    # Intercept-resend attack:
    # 1. Alice prepares a qubit.
    # 2. Eve measures it using her randomly chosen basis.
    # 3. Eve sends Bob a new qubit matching her measured result and basis.
    # 4. Bob measures using his chosen basis.

    # Alice prepares the original qubit.
    eve_circuit = QuantumCircuit(1, 1)
    if alice_bit == 1:
        eve_circuit.x(0)
    if alice_basis == 1:
        eve_circuit.h(0)

    # Eve measures in her chosen basis.
    if eve_basis == 1:
        eve_circuit.h(0)
    eve_circuit.measure(0, 0)

    compiled = transpile(eve_circuit, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    eve_result = int(list(result.get_counts().keys())[0])

    # Eve resends a newly prepared qubit based on what she measured.
    bob_circuit = QuantumCircuit(1, 1)
    if eve_result == 1:
        bob_circuit.x(0)
    if eve_basis == 1:
        bob_circuit.h(0)

    # Bob measures in his chosen basis.
    if bob_basis == 1:
        bob_circuit.h(0)
    bob_circuit.measure(0, 0)

    compiled = transpile(bob_circuit, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    bob_result = int(list(result.get_counts().keys())[0])

    return eve_result, bob_result

4. Alice chooses secret bits and bases

In [5]:
# Alice randomly chooses data bits and bases using quantum randomness.
alice_bits = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print("Alice bits: ", alice_bits)
print("Alice bases:", alice_bases)

Alice bits:  [0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1]
Alice bases: [1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1]


5. Eve chooses bases and prepares attack

In [6]:
# Eve randomly chooses bases and measures every qubit before Bob receives it.
eve_bases = quantum_random_bits(N)
eve_results = []

print("Eve bases:", eve_bases)

Eve bases: [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1]


6. Bob chooses bases, Eve intercepts each Qubit

In [7]:
# Bob randomly chooses measurement bases using quantum randomness.
bob_bases = quantum_random_bits(N)
bob_results = []

for i in range(N):
    eve_result, bob_result = alice_prepare_eve_measure_bob_measure(
        alice_bits[i],
        alice_bases[i],
        eve_bases[i],
        bob_bases[i]
    )
    eve_results.append(eve_result)
    bob_results.append(bob_result)

print("Bob bases:  ", bob_bases)
print("Eve results:", eve_results)
print("Bob results:", bob_results)

Bob bases:   [1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1]
Eve results: [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1]
Bob results: [0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1]


7. Public Comparison

In [8]:
matching_positions = []
for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        matching_positions.append(i)

sifted_alice_key = [alice_bits[i] for i in matching_positions]
sifted_bob_key = [bob_results[i] for i in matching_positions]

print("Matching positions:", matching_positions)
print("Alice sifted key: ", sifted_alice_key)
print("Bob sifted key:   ", sifted_bob_key)

Matching positions: [0, 2, 7, 8, 9, 10, 12, 13, 15, 17, 19, 20, 21, 23]
Alice sifted key:  [0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1]
Bob sifted key:    [0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1]


8. Reveal test bits and get error rate

In [9]:
# They reveal part of the sifted key to estimate the error rate.
test_size = max(1, math.floor(len(sifted_alice_key) * TEST_FRACTION))
test_alice = sifted_alice_key[:test_size]
test_bob = sifted_bob_key[:test_size]

errors = 0
for i in range(test_size):
    if test_alice[i] != test_bob[i]:
        errors += 1

error_rate = errors / test_size
attack_detected = error_rate > ATTACK_THRESHOLD

print("Test bits revealed:", test_size)
print("Errors in test bits:", errors)
print("Error rate:", round(error_rate, 3))
print("Attack detected?", attack_detected)

Test bits revealed: 7
Errors in test bits: 2
Error rate: 0.286
Attack detected? True


9. Decide whether to use or reject key

In [11]:
# The unrevealed part would only be used if no attack is detected.
final_alice_key = sifted_alice_key[test_size:]
final_bob_key = sifted_bob_key[test_size:]

print("BB84 WITH INTERCEPT-RESEND ATTACKER")
print("Qubits sent:", N)
print("Matching bases:", len(matching_positions))
print("Alice final key:", final_alice_key)
print("Bob final key:  ", final_bob_key)
print("Final keys match?", final_alice_key == final_bob_key)

if attack_detected:
    print("Protocol result: The error rate is too high, so Alice and Bob should not use this key.")
else:
    print("Protocol result: No attack detected in this run.")

BB84 WITH INTERCEPT-RESEND ATTACKER
Qubits sent: 24
Matching bases: 14
Alice final key: [1, 1, 1, 0, 0, 0, 1]
Bob final key:   [1, 0, 1, 0, 0, 1, 1]
Final keys match? False
Protocol result: The error rate is too high, so Alice and Bob should not use this key.
